# Portfolio risk and Treasury controls

These are executed preliminary results with retrospective explanation; full independent replication has not been established. Historical evaluation blocks were reused; they are not fresh holdouts.

Read the saved outputs without executing this notebook. Original code cells, their metadata, execution counts and outputs are preserved. This package supplies selected scientific evidence and context; the code retains its original source environment and frozen-input dependencies. It is not a standalone reproduction environment. See the [notebook index](README.md).


### Context: a new endpoint, not an aggregation of company forecasts

The target here is next-quarter **portfolio return variance**, scored primarily by QLIKE. P0=F, P1=F+SMH excess returns, P2=F+Treasury yield changes T, P3=F+T+SMH. All four retain identical holdings, weights, training dates and future target dates. Industry-group residual blocks are primary; diagonal residuals are the fixed sensitivity. This application uses daily-return covariance estimation, never C/R/L/S/V/A company forecasts.

T consists of changes in the FMP ten-year yield and ten-year-minus-two-year spread, in basis points. Both immediately adjacent equity-session endpoints must be directly present; no fill or holiday bridge. These are explanatory yield-change factors, not bond returns or causal policy shocks. FMP tenor-equivalent data are not certified identical to FRED DGS series or historical point-in-time vintages. The exact frozen JSON's later recovery concerns availability, not provider equivalence. Complete portfolio explanation.


## Research question: Bounded portfolio covariance application

Four matched covariance models ask whether SMH changes portfolio risk forecasts after Treasury controls. This is separate from company forecasts. All calculations below use actual frozen returns, original formation holdings and the same dates across each four-model comparison. Industry blocks are primary; diagonal residuals are the fixed sensitivity. QLIKE is lower-is-better. Previously inspected periods are reused historical evidence.


In [1]:
from pathlib import Path
import sys, json, hashlib
import numpy as np
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'extensions/company_risk').exists())
sys.path.insert(0, str(ROOT/'src'))
from quiet_index_500.foundation import loader, calendar
from quiet_index_500 import settings
from quiet_index_500.covariance_evaluation import quarterly_mean_uncertainty
# Original source is read-only; accepted processed files and correction ledger remain local.
loader.raw_data_folder = ROOT/'data/raw'
OUT=ROOT/'extensions/company_risk/results/portfolio';OUT.mkdir(parents=True,exist_ok=True)
F=['mkt_rf','smb','hml','rmw','cma','mom']
returns=loader.load_excess_returns(); factors=loader.load_factors()
factors['smh_excess']=loader.load_smh_excess_returns()
membership=pd.read_csv(ROOT/'results/covariance_evaluation/membership.csv',parse_dates=['formation_date'])
original=pd.read_csv(ROOT/'results/covariance_evaluation/quarterly_evaluation.csv',parse_dates=['formation_date'])
raw_path=ROOT/'extensions/company_risk/data/raw/treasury/fmp_treasury_rates.json'
yields=pd.DataFrame(json.loads(raw_path.read_text())).drop_duplicates('date').set_index('date')
yields.index=pd.to_datetime(yields.index); yields=yields.sort_index()[['year2','year10']].apply(pd.to_numeric)
# Reindex BEFORE differencing: an unobserved endpoint excludes the change, never creates zero.
y=yields.reindex(returns.index)
treasury=pd.DataFrame({'yield10_bp':100*y.year10.diff(), 'slope_bp':100*(y.year10-y.year2).diff()})
factors=factors.join(treasury)
coverage={'rows':len(yields),'start':str(yields.index.min().date()),'end':str(yields.index.max().date()),'valid_changes':int(treasury.notna().all(axis=1).sum()),'source_sha256':hashlib.sha256(raw_path.read_bytes()).hexdigest()}
coverage


{'rows': 3170,
 'start': '2014-01-02',
 'end': '2026-09-30',
 'valid_changes': 2880,
 'source_sha256': 'e9002194d2d51b29e522a86cda2f8079a7aa3510f2e9a79c6646953746373c9e'}

## Notebook-local covariance estimator and exact accounting

Intercept OLS makes fitted and residual returns orthogonal. All covariances use n−1. Adding one factor preserves individual variances; industry-block residuals also preserve every within-industry entry. The unrestricted fitted-plus-residual covariance reconstructs observed covariance. High/low contributions partition parent weights; their cross covariance is counted twice.


In [2]:
def fit_covariance(Y, X, weights, groups):
    design=np.column_stack([np.ones(len(X)), X])
    coef, _, rank, _=np.linalg.lstsq(design,Y,rcond=None)
    if rank!=design.shape[1]:raise ValueError('rank-deficient design')
    fitted=design@coef; residual=Y-fitted
    cov_f=np.cov(fitted,rowvar=False,ddof=1); cov_e=np.cov(residual,rowvar=False,ddof=1)
    masks={'industry_block':groups[:,None]==groups[None,:], 'diagonal':np.eye(len(weights),dtype=bool)}
    matrices={name:252*(cov_f+cov_e*mask) for name,mask in masks.items()}
    reconstruction=float(np.max(np.abs(252*(cov_f+cov_e-np.cov(Y,rowvar=False,ddof=1)))))
    assert reconstruction < 1e-10
    return {'matrices':matrices,'fitted':fitted,'residual':residual,'coef':coef,'reconstruction_error':reconstruction,
            'factor_variance':252*np.var(fitted@weights,ddof=1),
            'predictions':{name:float(weights@M@weights) for name,M in matrices.items()}}

def verify_one_factor_revision(base,extended,X,smh,weights,groups,high):
    design=np.column_stack([np.ones(len(X)),X]); u=smh-design@np.linalg.lstsq(design,smh,rcond=None)[0]
    load=u@base['residual']/(u@u)
    added=smh*(extended['coef'][-1]@weights)
    shared=extended['fitted']@weights-added
    shared_revision=252*np.var(shared,ddof=1)-base['factor_variance']
    added_variance=252*np.var(added,ddof=1)
    cross_factor=504*np.cov(shared,added,ddof=1)[0,1]
    assert abs(shared_revision+added_variance+cross_factor-(extended['factor_variance']-base['factor_variance']))<1e-10
    rows=[]
    for treatment,M in extended['matrices'].items():
        D=M-base['matrices'][treatment]
        labels=groups if treatment=='industry_block' else np.arange(len(weights))
        sums=np.array([np.sum(weights[labels==g]*load[labels==g]) for g in np.unique(labels)])
        identity=252*np.var(u,ddof=1)*((weights@load)**2-sums@sums)
        actual=float(weights@D@weights)
        wh=weights*high;wl=weights*(~high)
        hh=float(wh@D@wh);ll=float(wl@D@wl);cross=2*float(wh@D@wl)
        within=groups[:,None]==groups[None,:] if treatment=='industry_block' else np.eye(len(weights),dtype=bool)
        error=float(np.max(np.abs(D[within])))
        assert error<1e-10 and abs(actual-identity)<1e-10 and abs(actual-hh-ll-cross)<1e-10
        rows.append(dict(residual_treatment=treatment,revision=actual,one_factor_formula=identity,
                         identity_error=actual-identity,within_entry_error=error,high_contribution=hh,
                         low_contribution=ll,cross_contribution=cross,group_sum_error=actual-hh-ll-cross,
                         shared_factor_revision=shared_revision,added_factor_variance=added_variance,factor_cross_term=cross_factor,
                         factor_revision=extended['factor_variance']-base['factor_variance'],
                         residual_revision=actual-(extended['factor_variance']-base['factor_variance'])))
    return rows


### Why exact accounting does not guarantee a good forecast

On identical historical dates, intercept OLS reallocates fitted and residual variation when a factor is added. Retaining each residual variance preserves every total diagonal entry; retaining whole industry blocks also preserves every within-industry entry. Only cross-industry entries can change under that restriction. With unrestricted residual covariance, fitted plus residual covariance reconstructs the sample covariance entirely.

Parent variance is high contribution + low contribution + twice high/low covariance. In covariance_accounting.csv, `cross_contribution` already includes that doubling. These are the parent weights partitioned, not separately fitted portfolios. They explain changes in predicted annualized variance, never “shares of QLIKE loss.” QLIKE is nonlinear in forecast/realized variance.


## Original evidence bridge and matched Treasury application

The original bridge uses the original two-model common dates and checks its saved predictions. Treasury comparisons use a stricter four-model common training matrix, retaining original formation holdings. Future variance uses identical complete dates across all holdings and conventional/SMH factors, without requiring realized Treasury factors: the target is return variance, and no future explanatory factor is needed. Training and outcome date digests, omissions and counts are saved. No future quote changes holdings.


In [3]:
models={'P0':F,'P1':F+['smh_excess'],'P2':F+['yield10_bp','slope_bp'],'P3':F+['yield10_bp','slope_bp','smh_excess']}
records=[]; accounting=[];bridge=[];samples=[];date_ledger=[]
for formation, members in membership.query("window_sessions==504 and portfolio=='snapshot'").groupby('formation_date'):
    if calendar.outcome_block(formation)=='incomplete':continue
    members=members.set_index('ticker');w=members.weight.to_numpy();groups=members.gics_industry_group_code.to_numpy()
    train=calendar.estimation_window(returns.index,formation,504)
    future=calendar.following_quarter_sessions(returns.index,formation)
    Y=returns.reindex(index=train,columns=members.index)
    high=members.formation_smh_loading.to_numpy()>members.formation_smh_loading.median()
    oldmask=Y.notna().all(axis=1)&factors.reindex(train)[models['P1']].notna().all(axis=1)
    if oldmask.sum()>=400:
        oldY=Y.loc[oldmask].to_numpy(); oldX=factors.loc[train[oldmask]]
        oldfits={m:fit_covariance(oldY,oldX[models[m]].to_numpy(),w,groups) for m in ['P0','P1']}
        for model,fit in oldfits.items():
            for treatment,pred in fit['predictions'].items():
                saved=original.query("window_sessions==504 and portfolio=='snapshot'")
                saved=saved[(saved.formation_date==formation)&(saved.model=={'P0':'ff5_mom','P1':'ff5_mom_smh'}[model])&(saved.residual_treatment==treatment)]
                delta=pred-float(saved.predicted_variance.iloc[0]);assert abs(delta)<1e-10
                bridge.append(dict(formation_date=formation,model=model,residual_treatment=treatment,recomputed=pred,saved=float(saved.predicted_variance.iloc[0]),error=delta))
        for row in verify_one_factor_revision(oldfits['P0'],oldfits['P1'],oldX[F].to_numpy(),oldX.smh_excess.to_numpy(),w,groups,high):
            accounting.append(dict(formation_date=formation,sample='original',pair='P0_P1',**row))
    mask=Y.notna().all(axis=1)&factors.reindex(train)[models['P3']].notna().all(axis=1)
    futureY=returns.reindex(index=future,columns=members.index)
    valid=futureY.notna().all(axis=1)&factors.reindex(future)[models['P1']].notna().all(axis=1)
    for phase,dates,validity in [('training',train,mask),('outcome',future,valid)]:
        date_ledger.extend(dict(formation_date=formation,phase=phase,date=date,included=bool(ok)) for date,ok in zip(dates,validity))
    samples.append(dict(formation_date=formation,holdings=len(w),weight_sum=w.sum(),training_observations=int(mask.sum()),training_omissions=int((~mask).sum()),future_observations=int(valid.sum()),future_omissions=int((~valid).sum()),represented_snapshot_weight=members.snapshot_weight.sum(),training_dates_sha256=hashlib.sha256(','.join(train[mask].strftime('%Y-%m-%d')).encode()).hexdigest(),future_dates_sha256=hashlib.sha256(','.join(future[valid].strftime('%Y-%m-%d')).encode()).hexdigest()))
    if mask.sum()<400 or valid.sum()<40:continue
    X=factors.loc[train[mask]]; commonY=Y.loc[mask].to_numpy();realized=252*np.var(futureY.loc[valid].to_numpy()@w,ddof=1)
    fits={m:fit_covariance(commonY,X[names].to_numpy(),w,groups) for m,names in models.items()}
    for pair in [('P0','P1'),('P2','P3')]:
        for row in verify_one_factor_revision(fits[pair[0]],fits[pair[1]],X[models[pair[0]]].to_numpy(),X.smh_excess.to_numpy(),w,groups,high):
            accounting.append(dict(formation_date=formation,sample='treasury_matched',pair='_'.join(pair),**row))
    for model,fit in fits.items():
        for treatment,pred in fit['predictions'].items():
            ratio=realized/pred
            records.append(dict(formation_date=formation,outcome_quarter=str(calendar.outcome_quarter(formation)),outcome_block=calendar.outcome_block(formation),model=model,residual_treatment=treatment,predicted_variance=pred,realized_variance=realized,qlike=ratio-np.log(ratio)-1,factor_variance=fit['factor_variance'],residual_variance=pred-fit['factor_variance'],reconstruction_error=fit['reconstruction_error']))
pd.DataFrame(date_ledger).to_csv(OUT/'date_ledger.csv',index=False)
bridge=pd.DataFrame(bridge);accounting=pd.DataFrame(accounting);forecasts=pd.DataFrame(records);samples=pd.DataFrame(samples)
for name,table in [('original_bridge',bridge),('covariance_accounting',accounting),('forecasts',forecasts),('samples',samples)]:table.to_csv(OUT/f'{name}.csv',index=False)
print('Original saved predictions reconciled:',len(bridge),'Treasury forecasts:',len(forecasts))
samples


Original saved predictions reconciled: 132 Treasury forecasts: 240


,formation_date,holdings,weight_sum,training_observations,training_omissions,future_observations,future_omissions,represented_snapshot_weight,training_dates_sha256,future_dates_sha256
0,2017-12-29,461,1.0,486,18,61,0,0.954222,7302e209a445867848aa145142c5e7734a401afbce06fe...,a2c549cc41c3b219690d4ab349d80a444d5498627f256f...
1,2018-03-29,463,1.0,394,110,62,2,0.956940,b33077a4833a9c4be16139c40f5f9c8f1e852d721b52f2...,8b0ac8b0c9339f31700bf483d1b287747e83e8fa051b4c...
2,2018-06-29,464,1.0,420,84,63,0,0.957602,f05f57887d49bc6e840e057efcdaae28eeaac6ac22f852...,9af0b02fd55245e94489a4c534fb17672c1866c91bd427...
3,2018-09-28,466,1.0,410,94,63,0,0.959184,bcc29531a06bcb2402d3cf2a4456b378c29394363d931d...,b0c84b5dd7e9d9e483d96f555c1e701e4ef8b6972c52a7...
4,2018-12-31,468,1.0,397,107,61,0,0.960261,4afd1bf96d2b7b3a248de11d46e39fde127ffd3a1f1114...,ff77cfe6a680af6296dcddccc1078c1d90f0459ba039d8...
5,2019-03-29,468,1.0,458,46,63,0,0.960261,2a769bd8d1d185117d163a119a5e99a2c8777077e787ba...,b099cf7a0d9732c58c9736fa95c14574c9693f5f99954a...
6,2019-06-28,468,1.0,490,14,64,0,0.960261,8e0e9a57a1b8a1ab5971d4946609e0d11bbeb7bece175d...,9106648fc1c794195b7460db605c428b53054f76cabee6...
7,2019-09-30,469,1.0,428,76,56,8,0.960644,bc5180c99d44d495fb5da8fd06669f0e8119870f817ff3...,266e22f818239ba5ad1e0a11f13a0408b10efb39fcad45...
8,2019-12-31,469,1.0,478,26,59,3,0.960644,f8271218396679a489442ba0797637325a96e7fb569f1e...,4ffa719701c0ed69621d14b6c19e42798929f28671148c...
9,2020-03-31,470,1.0,398,106,63,0,0.961882,43022302a81f32413abf8f3044aa4add63988cc2c716f3...,d47f5186e970df0726d3fb14b06a7fbbc54dedc2648363...


### What the saved bridge and coverage establish

The source reconciles **132 original snapshot predictions** with maximum difference **8.60×10⁻¹⁴**, then writes **240 Treasury forecast rows** = 30 quarters × 4 models × 2 residual treatments.  Four development quarters fail the 400-common-training-observation rule. Actual matched periods are **16 development quarters within 2018Q1–2022Q4, 8 validation 2023–2024, 6 confirmation 2025Q1–2026Q2**. The sample ledger includes candidates as well as eligible formations; reconcile scored counts to forecasts.csv.

The frozen FMP response has **3,170 dates, 2014-01-02–2026-09-30**, yielding **2,880 valid directly observed equity-session changes**. Modern yield vintage and provider-methodology equivalence remain limitations. Future target dates require complete return/conventional/SMH observations, but not future Treasury factors: those are unnecessary to measure realized portfolio variance.

Original/Treasury-matched covariance identities pass 1e−10 absolute annualized covariance; largest within-industry discrepancy is about 2.85×10⁻¹⁴. This demonstrates accounting consistency, not next-quarter prediction success.


In [4]:
contrasts=[];summaries=[]
for (treatment,quarter,block), part in forecasts.groupby(['residual_treatment','outcome_quarter','outcome_block']):
    q=part.set_index('model').qlike
    contrasts.append(dict(residual_treatment=treatment,outcome_quarter=quarter,outcome_block=block,smh_without_treasury=q.P0-q.P1,smh_with_treasury=q.P2-q.P3,attenuation=(q.P0-q.P1)-(q.P2-q.P3),treasury_without_smh=q.P0-q.P2))
contrasts=pd.DataFrame(contrasts)
for (treatment,block),part in contrasts.groupby(['residual_treatment','outcome_block']):
    for contrast in ['smh_without_treasury','smh_with_treasury','attenuation','treasury_without_smh']:
        result=quarterly_mean_uncertainty(part[contrast],quarters=pd.PeriodIndex(part.outcome_quarter,freq='Q').to_timestamp())
        summaries.append(dict(residual_treatment=treatment,outcome_block=block,contrast=contrast,**result))
summary=pd.DataFrame(summaries)
contrasts.to_csv(OUT/'quarterly_contrasts.csv',index=False);summary.to_csv(OUT/'contrast_summary.csv',index=False)
(OUT/'completion.json').write_text(json.dumps({'status':'completed' if len(forecasts) else 'H7 deferred: inadequate directly observed endpoint coverage','treasury_coverage':coverage,'original_bridge_rows':len(bridge),'maximum_original_error':float(bridge.error.abs().max()),'forecast_rows':len(forecasts),'identity_tolerance':1e-10,'bootstrap':'2000 corrected circular replicates; block4 seed20261014; whole quarters; conditional on fitted exposures'},indent=2))
summary[['residual_treatment','outcome_block','contrast','quarters','mean_difference','circular_bootstrap_interval_low','circular_bootstrap_interval_high']]


,residual_treatment,outcome_block,contrast,quarters,mean_difference,circular_bootstrap_interval_low,circular_bootstrap_interval_high
0,diagonal,confirmation,smh_without_treasury,6,0.004553,-0.002248,0.011354
1,diagonal,confirmation,smh_with_treasury,6,0.004522,-0.002098,0.011143
2,diagonal,confirmation,attenuation,6,0.000031,-0.000154,0.000245
3,diagonal,confirmation,treasury_without_smh,6,-0.000088,-0.000157,-0.000036
4,diagonal,development,smh_without_treasury,16,0.020775,0.004199,0.039204
5,diagonal,development,smh_with_treasury,16,0.020794,0.004185,0.039289
6,diagonal,development,attenuation,16,-0.000019,-0.000272,0.000164
7,diagonal,development,treasury_without_smh,16,0.000239,-0.000129,0.000755
8,diagonal,validation,smh_without_treasury,8,-0.005006,-0.007227,-0.002784
9,diagonal,validation,smh_with_treasury,8,-0.004974,-0.007178,-0.002770


### Forecast results: absolute QLIKE differences

| Industry-block primary | SMH before Treasury | SMH after Treasury | Attenuation: before−after |
|---|---:|---:|---:|
| Development (16q) | −0.000467 | −0.000430 | −0.000037 |
| Validation (8q) | +0.003368 | +0.003378 | −0.000009 |
| Confirmation (6q) | −0.016927 | −0.016891 | −0.000037 |

contrast_summary.csv, `industry_block`, indicated `outcome_block`; `contrast=smh_without_treasury`, `smh_with_treasury`, `attenuation`. Positive favors the added factor; positive attenuation means Treasury reduces the SMH gain. These are absolute mean QLIKE differences, **not percentages explained away**. Confirmation attenuation uses unrounded rows (about −0.0000366).

![Portfolio increments, both residual restrictions and all reused blocks](assets/portfolio-risk-and-treasury-controls-portfolio_contrasts.png)

Read left for before/after gains, right for attenuation on a much smaller native-loss scale. Saved circular conditional 95% intervals span zero in primary confirmation: before [−0.045083,0.011228], after [−0.044841,0.011060], attenuation [−0.000301,0.000190].

Diagonal sensitivity gives confirmation **+0.004553** before and **+0.004522** after, both intervals crossing zero. Its sign reversal is important evidence that residual covariance treatment matters. Treasury barely changes either version. **Takeaway:** completed covariance identities coexist with uncertain and restriction-sensitive forecast results. No causal policy/duration conclusion follows. 07 keeps this application separate from company findings.


## Interpretation and limits

Positive QLIKE increments favor the added factor; positive attenuation means a smaller SMH increment after Treasury controls. These are absolute loss differences, not percentages explained away. Covariance identities verify arithmetic, not forecasting success. The high/low columns partition the parent portfolio and include cross covariance; they are not additional forecast portfolios. Treasury controls represent yield changes and slope changes, not bond returns or causal policy shocks. Modern historical FMP yields are not point-in-time vintages. The six-quarter confirmation block has already been inspected in the original study; intervals condition on fitted exposures and memberships. The main company investigation stands independently of these portfolio results.
